In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1) Generate Synthetic Datasets

In [ ]:
n_samples = 5000

# Credit card transactions
amount = np.random.gamma(shape=2.0, scale=80.0, size=n_samples)
transaction_hour = np.random.randint(0, 24, n_samples)
merchant_risk = np.random.uniform(0, 1, n_samples)
card_age_months = np.random.randint(1, 120, n_samples)
chargeback_count = np.random.poisson(0.2, n_samples)
foreign_txn = np.random.binomial(1, 0.15, n_samples)

fraud_signal = (
    0.004 * amount + 0.8 * merchant_risk + 0.5 * foreign_txn +
    0.3 * chargeback_count + 0.2 * ((transaction_hour < 6) | (transaction_hour > 22)).astype(int)
)
fraud_prob = 1 / (1 + np.exp(-(fraud_signal - 1.5)))
is_fraud = np.random.binomial(1, np.clip(fraud_prob, 0, 1))

credit_df = pd.DataFrame({
    'amount': amount,
    'transaction_hour': transaction_hour,
    'merchant_risk': merchant_risk,
    'card_age_months': card_age_months,
    'chargeback_count': chargeback_count,
    'foreign_txn': foreign_txn,
    'is_fraud': is_fraud
})

# Loan applicants
annual_income = np.random.normal(65000, 20000, n_samples).clip(12000, 250000)
loan_amount = np.random.normal(18000, 9000, n_samples).clip(1000, 100000)
credit_score = np.random.normal(680, 70, n_samples).clip(300, 850)
debt_to_income = np.random.uniform(0.05, 0.75, n_samples)
employment_years = np.random.randint(0, 35, n_samples)
past_due_count = np.random.poisson(0.4, n_samples)

loan_signal = (
    -0.008 * (credit_score - 650) + 1.7 * debt_to_income + 0.4 * past_due_count +
    0.00002 * loan_amount - 0.00001 * annual_income - 0.03 * employment_years
)
default_prob = 1 / (1 + np.exp(-loan_signal))
defaulted = np.random.binomial(1, np.clip(default_prob, 0, 1))

loan_df = pd.DataFrame({
    'annual_income': annual_income,
    'loan_amount': loan_amount,
    'credit_score': credit_score,
    'debt_to_income': debt_to_income,
    'employment_years': employment_years,
    'past_due_count': past_due_count,
    'defaulted': defaulted
})

credit_df.head(), loan_df.head()

## 2) Credit Card Fraud Detection

In [ ]:
X_fraud = credit_df.drop(columns=['is_fraud'])
y_fraud = credit_df['is_fraud']

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fraud, y_fraud, test_size=0.25, random_state=42, stratify=y_fraud
)

fraud_scaler = StandardScaler()
Xf_train_sc = fraud_scaler.fit_transform(Xf_train)
Xf_test_sc = fraud_scaler.transform(Xf_test)

fraud_model = LogisticRegression(max_iter=200, class_weight='balanced')
fraud_model.fit(Xf_train_sc, yf_train)

yf_prob = fraud_model.predict_proba(Xf_test_sc)[:, 1]
yf_pred = (yf_prob >= 0.5).astype(int)

print('Fraud ROC-AUC:', roc_auc_score(yf_test, yf_prob))
print(classification_report(yf_test, yf_pred))

## 3) Loan Default Prediction + Customer Risk Scoring

In [ ]:
X_loan = loan_df.drop(columns=['defaulted'])
y_loan = loan_df['defaulted']

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_loan, y_loan, test_size=0.25, random_state=42, stratify=y_loan
)

loan_scaler = StandardScaler()
Xl_train_sc = loan_scaler.fit_transform(Xl_train)
Xl_test_sc = loan_scaler.transform(Xl_test)

loan_model = LogisticRegression(max_iter=200, class_weight='balanced')
loan_model.fit(Xl_train_sc, yl_train)

yl_prob = loan_model.predict_proba(Xl_test_sc)[:, 1]
yl_pred = (yl_prob >= 0.5).astype(int)

print('Loan Default ROC-AUC:', roc_auc_score(yl_test, yl_prob))
print(classification_report(yl_test, yl_pred))

# Customer Risk Scoring (0-100)
risk_scores = (loan_model.predict_proba(loan_scaler.transform(X_loan))[:, 1] * 100).round(2)
risk_df = X_loan.copy()
risk_df['risk_score'] = risk_scores
risk_df.head()

## 4) Transaction Anomaly Detection

In [ ]:
anomaly_features = credit_df.drop(columns=['is_fraud'])
anom_scaler = StandardScaler()
Xa = anom_scaler.fit_transform(anomaly_features)

iso = IsolationForest(contamination=0.03, random_state=42)
anom_labels = iso.fit_predict(Xa)
anom_scores = -iso.score_samples(Xa)

anomaly_df = anomaly_features.copy()
anomaly_df['anomaly_label'] = (anom_labels == -1).astype(int)
anomaly_df['anomaly_score'] = anom_scores

print('Detected anomaly rate:', anomaly_df['anomaly_label'].mean())
anomaly_df.sort_values('anomaly_score', ascending=False).head(10)

## 5) Spending Pattern Clustering

In [ ]:
cluster_features = credit_df[['amount', 'transaction_hour', 'merchant_risk', 'chargeback_count', 'foreign_txn']].copy()

cl_scaler = StandardScaler()
Xc = cl_scaler.fit_transform(cluster_features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(Xc)
sil = silhouette_score(Xc, cluster_labels)

cluster_df = cluster_features.copy()
cluster_df['cluster'] = cluster_labels
cluster_profile = cluster_df.groupby('cluster', as_index=False).mean(numeric_only=True)

print('Silhouette score:', sil)
cluster_profile

## Save Outputs

In [ ]:
risk_df.to_csv('outputs/customer_risk_scores.csv', index=False)
cluster_profile.to_csv('outputs/cluster_profile_summary.csv', index=False)
print('Saved: outputs/customer_risk_scores.csv and outputs/cluster_profile_summary.csv')